# Оценка текстов

In [5]:
from pathlib import Path
import pandas as pd

data = pd.read_csv(Path.cwd().parent / "data/dataset_maxi_literal.csv")
data.info()

example_original_texts = data[data['text_domen'] == "художественный"][:5]
exmple_literal_texts =  data[data['text_domen'] == "новостной"][:5]

<class 'pandas.DataFrame'>
RangeIndex: 84 entries, 0 to 83
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Unnamed: 0             84 non-null     int64  
 1   author                 54 non-null     str    
 2   source                 84 non-null     str    
 3   link                   42 non-null     str    
 4   year_of_creation       84 non-null     int64  
 5   text_domen             84 non-null     str    
 6   original_text          84 non-null     str    
 7   original_text_clean    84 non-null     str    
 8   idiomatic_expressions  84 non-null     str    
 9   literal_version        84 non-null     str    
 10  len                    84 non-null     int64  
 11  word_count             84 non-null     int64  
 12  ratio                  84 non-null     float64
 13  idiom_count            84 non-null     int64  
 14  ratio_2                84 non-null     str    
dtypes: float64(1), int6

## Оценка качества генерации (сохранение контекста)

**Используемые метрики:**
1. BERTScore -- cosine similarity between embeddings
2. BLEURT -- enhanced BERTScore (more alligned with human evalation)
3. COMET -- аналог BLEURT
4. METEOR -- surface level matching (based on BLEU and ROUGE)
5. Word Mover's Distance (WMD) -- хорошо коррелирует с человеками(https://ar5iv.labs.arxiv.org/html/2004.05001)
---
Похожий подход использовался здесь: https://arxiv.org/pdf/2209.01835

А тут (вроде бы) говорится про высокое соответствие с человеческой анотацией: https://ieeexplore.ieee.org/ielx7/6287639/10380310/10522667.pdf?tp=&arnumber=10522667&isnumber=10380310&ref=aHR0cHM6Ly9pZWVleHBsb3JlLmllZWUub3JnL3N0YW1wL3N0YW1wLmpzcD9hcm51bWJlcj0xMDUyMjY2Nw==#3%233

### Имплементация

In [ ]:
from bert_score import score as bert_score_func
def measure_bertscore(references: list[str], candidates: list[str]) -> dict:
    """Вычисляет BERTScore (Precision, Recall, F1).
    
    Для русского языка используется xlm-roberta-base.
    rescale_with_baseline=True нормализует оценку, делая её более наглядной.
    """
    P, R, F1 = bert_score_func(
        cands=candidates, 
        refs=references, 
        lang="ru", 
        model_type="xlm-roberta-base",
        rescale_with_baseline=True
    )
    
    # Возвращаем средние значения по всему батчу текстов
    return {
        "bertscore_precision": P.mean().item(),
        "bertscore_recall": R.mean().item(),
        "bertscore_f1": F1.mean().item()
    }

In [ ]:
import os
from bleurt import score as bleurt_score
def measure_bluert(references: list[str], candidates: list[str], checkpoint_path: str = "./BLEURT-20") -> dict:
    """Вычисляет BLEURT score на основе мультиязычной модели BLEURT-20.
    
    Перед запуском нужно скачать архив модели BLEURT-20 и распаковать в рабочую директорию.
    """
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Чекпоинт BLEURT не найден по пути {checkpoint_path}. Скачайте и распакуйте BLEURT-20.")
        
    scorer = bleurt_score.BleurtScorer(checkpoint_path)
    scores = scorer.score(references=references, candidates=candidates)
    
    return {
        "bluert_score": sum(scores) / len(scores)
    }


In [ ]:
from comet import download_model, load_from_checkpoint
def measure_comet(references: list[str], candidates: list[str], sources: list[str] = None) -> dict:
    """Вычисляет COMET score (стандарт wmt22-comet-da).
    
    Если исходных текстов (sources) нет, передаем вместо них references, 
    чтобы модель могла выполнить синтаксическую проверку.
    """
    # Модель скачается автоматически при первом запуске
    model_path = download_model("Unbabel/wmt22-comet-da")
    model = load_from_checkpoint(model_path)
    
    if sources is None:
        sources = references
        
    # Форматируем данные под требования библиотеки
    data = [
        {"src": src, "mt": cand, "ref": ref}
        for src, cand, ref in zip(sources, candidates, references)
    ]
    
    model_output = model.predict(data, batch_size=8, gpus=1)
    
    return {
        "comet_score": model_output.system_score
    }

In [ ]:
import nltk
from evaluate import load

# Необходимые ресурсы NLTK для токенизации и стемминга
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)

def measure_meteor(references: list[str], candidates: list[str]) -> dict:
    """Вычисляет METEOR score через стандартный интерфейс Hugging Face Evaluate."""
    meteor_metric = load('meteor')
    results = meteor_metric.compute(predictions=candidates, references=references)
    
    return {
        "meteor_score": results["meteor"]
    }

In [ ]:
def measure_all_metrics(references: list, candidates: list, sources: list = None) -> dict:
    results = {}
    
    return results

## Оценка идиоматичности

**Используемые метрики:**

- Perplexity -- стандартная штука
- Surpisal (модели скармливают только левый контекст и смотрят на вероятности исследуемых токенов) -- довольно часто встречается, конкретно тут может говорить про новизну метафоры. коррелирует с оценками аннотаторов \(https://www.alphaxiv.org/abs/2601.02015) -- как можно сделать?
- Embedding-Based Concreteness Direction - sota подход (https://arxiv.org/pdf/2604.18296)
- cosine similariy (между двумя текстами)

*Лексические*
1. частота слова/тональность/субъективность
2. concreteness/abstractness -- но пока не понятно как считать (https://www.hse.ru/data/xf/907/469/1482/FINAL_TEXT_3.pdf) 

*Синтаксические*
1. Selectional Preference Violation (сочетаемость сущ + глагол, опять косинусное расстояние)
2. Парсить синтаксические деревья и искать конструкции/Tree Edit Distance (звучит забавно, но сложно и никто кажется не делает)

*Семантические*
1. Semantic Similarity + Linear Semantic Coherence (еще просто coherence) \
дистрибутивная фича для метафор, зависимость между глаголом и его предикатами + глаголом и окружением. если усреднить для двух текстов, может дать интересный результат, но может быть сложно интерпретировать \
(https://www.hse.ru/data/2019/08/30/1535711894/Distributional%20semantic%20features%20in%20Russian%20verbal%20metaphor%20identification.pdf) 
2. MelBERT (контекстуализированное/исходное представление слова) (скорее нет, подумать)
3. готовые классификаторы для детекции (правда, лучше ли они чем запромптить ллм, я не знаю) (не брать)

*Дискурсивные*
1. выделить дискурсивные формулы? (не брать)

*Прагматические*
1. sentiment analysis -- слова с противоположной тональностью в одном предложении -> возможно ирония (если будет время)

Кроме того:
- тут есть некоторые фичи (например про эллипсис) (https://arxiv.org/pdf/2309.12810) (может быть не показательно из-за размера)
- простая pos-статистика (подумать, сравнить что-то конктреное)
- не знаю, наверное неприменимо (ContrastWSD: Enhancing Metaphor Detection with Word Sense Disambiguation Following the Metaphor Identification Procedure) (не брать)

## LLM as a judge

1. Естественность употребления идиом (от 1 до 10 или похожее).
2. Образность текста (низкая, высокая, средняя) + индекс.
3. Можно попросить выделить идиомы из текста и проверить согласованность разметки на примерах датасета.
4. fluency, оценить грамматичность, найти языковые ошибки.